# Saving and Exporting

Once you have a CP you like, you'll want to send it somewhere — a plotter, a folder simulator, or a 3D printer. This notebook covers the export formats `pleat` ships with:

- `.heg` — pleat's native YAML serialization.
- `.fold` — the standard [FOLD](https://github.com/edemaine/fold) interchange format.
- SVG — In two flavours, one for pretty pictures, one optimized for laser cutters / pen plotters.
- A high-level `overlap.save_results` that writes a whole result directory in one call.

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import pleat
from pleat.rendering import multi_show, CREASE_PATTERN_PRESET

import tempfile, os

In [ ]:
G = pleat.example_graphs.from_tiles(pleat.example_tilesets.platonic(n=6), rings=2)
pleat.shrink_rotate.crease_orientation.assign_this_way_from_center(G)
G = pleat.shrink_rotate.shrink_rotate_pattern(G, alpha=0.6, factor=0.5)


## .heg save

Pleat's native YAML serialization captures the full graph structure plus arbitrary attributes.

Note: the load path currently uses `yaml.SafeLoader`, so graphs with non-trivial Python attributes (tuples, numpy scalars) save but don't round-trip.

In [ ]:
from pleat import io

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'pattern.heg')
    io.save_graph(path, G)
    size = os.path.getsize(path)
    print(f'wrote {size} bytes')

    # load it back
    G2 = io.load_graph(path)

multi_show(
    [G, G2],
    titles=['original', 'loaded'],
    **CREASE_PATTERN_PRESET
)


## Export via `G.save(...)`

`G.save('out')` writes all of `out.svg`, `out.png`, `out.heg` and `out.fold`. If you only want to save a subset, call with the appropriate file ending, e.g. `G.save('out.svg')`.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    out = os.path.join(d, 'pattern')
    G.save(out, **CREASE_PATTERN_PRESET)
    print('files in temp dir:', sorted(os.listdir(d)))
    print('SVG head:')
    print(open(out + '.svg').read()[:200])

## Plotter-ready SVG via `SvgwriteRenderer`

For laser-cutter / pen-plotter pipelines the dedicated `SvgwriteRenderer` produces an SVG split into `{name}_borders.svg` / `{name}_interior.svg` (so you can use different tool heads for cut vs. score).

These can then e.g. be converted to `.hpgl` with inkscape for plotting, see `plot_cp.py`.

In [ ]:
from pleat.rendering import SvgwriteRenderer

with tempfile.TemporaryDirectory() as d:
    out = os.path.join(d, 'pattern')
    renderer = SvgwriteRenderer() 
    renderer.render_graph(out + '.svg', G)
    print('files in temp dir:', sorted(os.listdir(d)))
    print('SVG head:')
    print(open(out + '.svg').read()[:200])


## All-in-one with `overlap.save_results`

If you've gone through `fold_complete` (demonstrated in the [Shrink-Rotate notebook](Shrink_Rotate_Tessellations.ipynb)), `save_results(result, path)` writes the CP, both folded views, a back-lit composite, and a plotter-ready SVG in one call.

## FOLD format

[FOLD](https://github.com/edemaine/fold) is the standard origami interchange
format. `save_fold` writes a `.fold` file (crease pattern with M/V/B assignments
and fold angles); `load_fold` reads one back.

In [ ]:
from pleat.io import save_fold, load_fold


with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'pattern.fold')
    save_fold(path, G)
    # cp.save(path) # also works
    print('wrote', os.path.getsize(path), 'bytes of FOLD')
    G_loaded = load_fold(path)


G_loaded.show()